# Phase 01.04 — LoRA r16 evaluation (F1)

Loads the pinned base model plus the saved r16 adapter and evaluates the exact frozen validation membership used by zero-shot.


In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace/RoadBuddy")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("CC", "/usr/bin/gcc")
os.environ.setdefault("CXX", "/usr/bin/g++")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from roadbuddy_common import *

os.chdir(PROJECT_ROOT)
seed_everything(SEED)
ensure_dirs()
print("Project root:", PROJECT_ROOT)
print("Model revision:", MODEL_REVISION)


Project root: /workspace/RoadBuddy
Model revision: b98f263eab246eb5269ade64edbdca8a887dc44d


In [2]:
from peft import PeftModel

DEBUG_LIMIT = 20  # Set None for final evaluation.
ADAPTER_RUN = "debug20" if DEBUG_LIMIT else "full"
ADAPTER_DIR = PATHS.lora / ADAPTER_RUN / "checkpoints" / "adapter_final"
VALIDATION_CSV = PATHS.phase1_split / "validation.csv"

assert ADAPTER_DIR.is_dir(), f"Adapter not found: {ADAPTER_DIR}"
val_df = pd.read_csv(VALIDATION_CSV)
frozen_ids = json.loads((PATHS.phase1_split / "validation_sample_ids.json").read_text(encoding="utf-8"))
assert sorted(val_df.sample_id.astype(str)) == frozen_ids


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load base + adapter


In [3]:
base_model, tokenizer = load_model_and_tokenizer(training=False)
base_model.img_context_token_id = tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
assert model.base_model.model.template == "Hermes-2"
print("Adapter:", ADAPTER_DIR)


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


Adapter: /workspace/RoadBuddy/outputs/phase01/lora_r16_f1/debug20/checkpoints/adapter_final


## Evaluate and save


In [4]:
predictions, metrics = evaluate_rows(model, tokenizer, val_df, limit=DEBUG_LIMIT)
run_name = "debug20" if DEBUG_LIMIT else "full"
out_dir = PATHS.lora / run_name / "evaluation"
out_dir.mkdir(parents=True, exist_ok=True)
predictions.to_csv(out_dir / "validation_predictions.csv", index=False)
save_json(out_dir / "validation_metrics.json", metrics)
save_json(out_dir / "evaluation_config.json", {
    "model_id": MODEL_ID, "revision": MODEL_REVISION, "adapter": str(ADAPTER_DIR),
    "validation_ids": str(PATHS.phase1_split / "validation_sample_ids.json"),
    "seed": SEED, "frames": 1, "flash_attention": False, "limit": DEBUG_LIMIT,
})
display(metrics)
display(predictions.head())
print("Saved:", out_dir)


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


{'rows': 20,
 'accuracy': 0.6,
 'macro_f1': 0.5694444444444444,
 'parse_rate': 1.0}

,sample_id,group_id,question_type,answer,prediction,raw_response,correct,num_tiles
0,train_0006,464458b4_386_clip_006_0039_0048_Y,unknown,B,B,B,True,7
1,train_0040,81392443_006_clip_005_0028_0035_Y,unknown,B,B,B,True,7
2,train_0041,81392443_006_clip_005_0028_0035_Y,unknown,B,B,B,True,7
3,train_0042,a5cb63a2_035_clip_001_0000_0007_Y,unknown,D,B,B,False,7
4,train_0043,a5cb63a2_035_clip_001_0000_0007_Y,unknown,B,B,B,True,7


Saved: /workspace/RoadBuddy/outputs/phase01/lora_r16_f1/debug20/evaluation


## Nhận xét sau lần chạy Phase 01.04

**Phạm vi:** đánh giá adapter debug trên cùng 20 validation rows của zero-shot baseline.

### Kết quả

| Chỉ số | LoRA r16 | Zero-shot | Chênh lệch LoRA − Zero |
|---|---:|---:|---:|
| Accuracy | 0.6000 | 0.6500 | -0.0500 |
| Macro-F1 | 0.5694 | 0.5985 | -0.0290 |
| Parse rate | 1.0000 | 1.0000 | 0.0000 |

LoRA đúng 12/20 và sai 8/20. F1 theo lớp: A=0.4444, B=0.6667, C=0.5000, D=0.6667. So với zero-shot, suy giảm tập trung ở lớp B: recall giảm từ 0.7273 xuống 0.6364. Các lỗi LoRA nằm tại `train_0042`, `train_0050`, `train_0051`, `train_0058`, `train_0060`, `train_0062`, `train_0071`, `train_0089`.

### Nhận định

Adapter được load và inference đúng, parse rate hoàn hảo, nhưng debug training không tạo cải thiện chất lượng. Trong paired comparison, chỉ một prediction thay đổi: `train_0062` từ B đúng ở zero-shot sang A sai ở LoRA. Không có trường hợp wrong-to-right.

Kết quả này phù hợp với phạm vi training cực ngắn 2 optimizer steps. Nó không chứng minh LoRA r16 kém về nguyên tắc; nó chỉ cho thấy adapter debug hiện tại chưa học được tín hiệu hữu ích và không nên dùng làm checkpoint cuối.

### Quyết định đề xuất

Giữ zero-shot làm baseline/winner tạm thời. Chỉ đánh giá lại LoRA sau full training với logging/checkpoint rõ ràng và trên đủ 298 validation rows.